In [3]:
import pandas as pd

df = pd.read_csv(r"D:\forecasting_system\data\processed\initial_clean.csv")

df.head()

,State,Date,Total,Category
0,Alabama,2019-01-12,109574036.0,Beverages
1,Alabama,2019-03-11,112189103.8,Beverages
2,Alabama,2019-06-10,129106730.4,Beverages
3,Alabama,2019-08-12,108083723.8,Beverages
4,Alabama,2019-10-11,110932912.8,Beverages


In [5]:
df['Date'] = pd.to_datetime(df['Date'])
print(df.dtypes)

State                  str
Date        datetime64[us]
Total              float64
Category               str
dtype: object


In [6]:
print(df.isnull().sum())

State       0
Date        0
Total       0
Category    0
dtype: int64


In [7]:
states = df['State'].unique()

for state in states:
    
    state_df = df[df['State'] == state]
    
    full_range = pd.date_range(
        start=state_df['Date'].min(),
        end=state_df['Date'].max(),
        freq='D'
    )
    
    missing_dates = full_range.difference(state_df['Date'])
    
    print(f"{state} -> Missing Dates: {len(missing_dates)}")

Alabama -> Missing Dates: 1599
Arizona -> Missing Dates: 1599
Arkansas -> Missing Dates: 1599
California -> Missing Dates: 1599
Colorado -> Missing Dates: 1599
Connecticut -> Missing Dates: 1599
Florida -> Missing Dates: 1599
Georgia -> Missing Dates: 1599
Illinois -> Missing Dates: 1599
Indiana -> Missing Dates: 1599
Iowa -> Missing Dates: 1599
Kansas -> Missing Dates: 1599
Kentucky -> Missing Dates: 1599
Louisiana -> Missing Dates: 1599
Maine -> Missing Dates: 1599
Maryland -> Missing Dates: 1599
Massachusetts -> Missing Dates: 1599
Michigan -> Missing Dates: 1599
Minnesota -> Missing Dates: 1599
Mississippi -> Missing Dates: 1599
Missouri -> Missing Dates: 1599
Nebraska -> Missing Dates: 1599
Nevada -> Missing Dates: 1599
New Hampshire -> Missing Dates: 1599
New Mexico -> Missing Dates: 1599
New York -> Missing Dates: 1599
North Carolina -> Missing Dates: 1599
Ohio -> Missing Dates: 1599
Oklahoma -> Missing Dates: 1599
Oregon -> Missing Dates: 1599
Pennsylvania -> Missing Dates: 159

In [8]:
all_states_data = []

states = df['State'].unique()

for state in states:
    
    state_df = df[df['State'] == state].copy()
    
    state_df = state_df.set_index('Date')
    
    full_range = pd.date_range(
        start=state_df.index.min(),
        end=state_df.index.max(),
        freq='D'
    )
    
    state_df = state_df.reindex(full_range)
    
    state_df['State'] = state
    
    state_df = state_df.rename_axis('Date').reset_index()
    
    all_states_data.append(state_df)

df = pd.concat(all_states_data, ignore_index=True)

In [9]:
print(df.isnull().sum())

Date            0
State           0
Total       68757
Category    68757
dtype: int64


In [11]:
print(df.columns)

Index(['Date', 'State', 'Total', 'Category'], dtype='str')


In [12]:
df['Total'] = df.groupby('State')['Total'].transform(
    lambda x: x.interpolate(method='linear')
)

In [13]:
df['Total'] = df.groupby('State')['Total'].transform(
    lambda x: x.ffill().bfill()
)

In [15]:
print(df.isnull().sum())

Date            0
State           0
Total           0
Category    68757
Sales           0
dtype: int64


In [16]:
df = df.sort_values(['State', 'Date'])

In [17]:
print(df.head())

print(df.shape)

print(df['State'].nunique())

        Date    State         Total   Category         Sales
0 2019-01-12  Alabama  1.095740e+08  Beverages  1.095740e+08
1 2019-01-13  Alabama  1.096191e+08        NaN  1.096191e+08
2 2019-01-14  Alabama  1.096642e+08        NaN  1.096642e+08
3 2019-01-15  Alabama  1.097093e+08        NaN  1.097093e+08
4 2019-01-16  Alabama  1.097544e+08        NaN  1.097544e+08
(76841, 5)
43


In [24]:
from pathlib import Path

BASE_DIR = Path.cwd().parent
data_path = BASE_DIR / "data" / "processed"

df.to_csv(data_path / "final_processed.csv", index=False)
df.to_csv("../data/processed/final_processed.csv", index=False)